# Market-making simulator demonstration

This notebook consumes completed package outputs. It does not contain a duplicate matching or backtesting engine. All provided runs use synthetic data and are demonstrations, not evidence of profitability.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import polars as pl

RUNS = {
    "fixed": Path("../runs/fixed-spread-demo"),
    "inventory": Path("../runs/inventory-demo"),
    "microprice": Path("../runs/microprice-demo"),
}
missing = [str(path) for path in RUNS.values() if not (path / "metrics.json").exists()]
if missing:
    raise FileNotFoundError(f"Run the README backtests first; missing: {missing}")

In [ ]:
rows = []
for name, path in RUNS.items():
    metrics = json.loads((path / "metrics.json").read_text())
    rows.append({
        "strategy": name,
        "net_pnl": metrics["pnl"]["net_pnl"],
        "gross_pnl": metrics["pnl"]["gross_pnl"],
        "fills": metrics["trading_activity"]["fill_events"],
        "end_inventory": metrics["inventory"]["end_of_session"],
        "max_drawdown": metrics["pnl"]["maximum_drawdown"],
    })
comparison = pl.DataFrame(rows)
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, path in RUNS.items():
    pnl = pl.read_parquet(path / "pnl.parquet")
    inventory = pl.read_parquet(path / "inventory.parquet")
    axes[0].plot(pnl["timestamp_ns"], pnl["net_pnl"], label=name)
    axes[1].plot(inventory["timestamp_ns"], inventory["inventory"], label=name)
axes[0].set(title="Net marked P&L (synthetic demonstration)", xlabel="timestamp ns")
axes[1].set(title="Inventory", xlabel="timestamp ns")
for ax in axes:
    ax.legend(); ax.grid(alpha=0.25)
plt.tight_layout()

## Interpretation limits

Compare mechanics, inventory control, message activity, cost reconciliation, and markouts—not synthetic profit alone. Level 2 queue position is approximate; historical markets do not react to simulated orders; marketable depth is approximate; and results are sensitive to costs, latency, cancellation allocation, and data quality. Annualized Sharpe is intentionally omitted for this event-time demonstration.